In [ ]:
import scanpy as sc
print(sc.__version__)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.io
import gzip
import os

def load_sample_manual(path, sample_name):
    # Load matrix
    matrix = scipy.io.mmread(os.path.join(path, 'matrix.mtx.gz')).T.tocsr()
    
    # Load barcodes
    with gzip.open(os.path.join(path, 'barcodes.tsv.gz'), 'rt') as f:
        barcodes = [line.strip() for line in f]
    
    # Load features — only 2 columns, take gene symbols (column 1)
    with gzip.open(os.path.join(path, 'features.tsv.gz'), 'rt') as f:
        features = [line.strip().split('\t') for line in f]
    
    gene_ids = [f[0] for f in features]
    gene_symbols = [f[1] for f in features]
    
    # Build AnnData
    adata = sc.AnnData(X=matrix)
    adata.obs_names = barcodes
    adata.var_names = gene_symbols
    adata.var['gene_ids'] = gene_ids
    adata.obs['sample'] = sample_name
    adata.obs['condition'] = 'PAIVS' if 'PAIVS' in sample_name else 'Control'
    
    return adata

samples = {
    'Control1': 'data/Control1_hCM',
    'Control2': 'data/Control2_hCM',
    'PAIVS1':   'data/PAIVS1_hCM',
    'PAIVS2':   'data/PAIVS2_hCM'
}

adatas = {}
for sample_name, path in samples.items():
    adata = load_sample_manual(path, sample_name)
    adatas[sample_name] = adata
    print(f"{sample_name}: {adata.shape[0]} cells, {adata.shape[1]} genes")

In [ ]:
import anndata
print(anndata.__version__)

In [ ]:
# Make gene names unique within each sample first
for sample_name, adata in adatas.items():
    adata.var_names_make_unique()
    print(f"{sample_name} var names unique: {adata.var_names.is_unique}")

In [ ]:
import anndata as ad

# Concatenate all samples into one AnnData object
adata_combined = ad.concat(adatas, label='sample')
adata_combined.var_names_make_unique()

print(adata_combined)

In [ ]:
adata_combined.write('data/combined_raw.h5ad')
print("Saved!")

Quality Control

QC metrics: mitochondrial %, total counts, genes per cell
Filter low quality cells and doublets
This is where bad projects fail — we'll do it properly

In [ ]:
# Calculate QC metrics
# Flag mitochondrial genes - they start with 'MT-'
adata_combined.var['mt'] = adata_combined.var_names.str.startswith('MT-')

sc.pp.calculate_qc_metrics(
    adata_combined,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print(adata_combined.obs.columns.tolist())

In [ ]:
import matplotlib.pyplot as plt

sc.pl.violin(adata_combined, 
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4, 
    multi_panel=True,
    save='_qc_before_filtering.png'
)

In [ ]:
# Filter cells
sc.pp.filter_cells(adata_combined, min_genes=200)
sc.pp.filter_genes(adata_combined, min_cells=3)

# Remove high MT% cells and doublets
adata_combined = adata_combined[
    (adata_combined.obs.pct_counts_mt < 10) &
    (adata_combined.obs.n_genes_by_counts < 6000)
].copy()

print(f"Cells remaining after QC: {adata_combined.n_obs}")
print(f"Genes remaining: {adata_combined.n_vars}")

In [ ]:
adata_combined.write('data/combined_filtered.h5ad')
print("Filtered data saved!")

Normalization

In [ ]:
# Normalize each cell to 10,000 total counts
sc.pp.normalize_total(adata_combined, target_sum=1e4)

# Log transform
sc.pp.log1p(adata_combined)

print("Normalization complete!")
print(adata_combined)

In [ ]:
sc.pp.highly_variable_genes(
    adata_combined,
    min_mean=0.0125,
    max_mean=3,
    min_disp=0.5
)

print(f"Highly variable genes: {adata_combined.var.highly_variable.sum()}")

sc.pl.highly_variable_genes(adata_combined, save='_hvg.png')

In [ ]:
sc.pp.scale(adata_combined, max_value=10)
sc.tl.pca(adata_combined, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_combined, log=True, save='_pca.png')

In [ ]:
pip install igraph leidenalg

In [ ]:
sc.pp.neighbors(adata_combined, n_neighbors=10, n_pcs=15)
sc.tl.umap(adata_combined)
sc.tl.leiden(adata_combined, resolution=0.5)

sc.pl.umap(adata_combined, 
    color=['leiden', 'condition', 'sample'],
    save='_clusters.png'
)

In [ ]:
sc.tl.rank_genes_groups(
    adata_combined, 
    groupby='condition',
    method='wilcoxon'
)

sc.pl.rank_genes_groups(
    adata_combined, 
    n_genes=25,
    sharey=False,
    save='_marker_genes.png'
)

In [ ]:
# Visualize key genes on UMAP
genes_of_interest = ['MT-ND1', 'MT-ND3', 'SPINK1', 'S100A6', 'ACTB']

sc.pl.umap(
    adata_combined,
    color=genes_of_interest,
    save='_gene_expression.png'
)

In [ ]:
# Find marker genes for each cluster
sc.tl.rank_genes_groups(
    adata_combined,
    groupby='leiden',
    method='wilcoxon'
)

# Show top 5 genes per cluster
sc.pl.rank_genes_groups_heatmap(
    adata_combined,
    n_genes=5,
    groupby='leiden',
    save='_cluster_markers.png'
)

In [ ]:
# Known cardiomyocyte state markers
marker_genes = {
    'Contractile': ['MYH7', 'TNNT2', 'MYH6', 'ACTC1'],
    'Stressed': ['HSPA1A', 'HSPB1', 'HSP90AA1'],
    'Mitochondrial': ['MT-ND1', 'MT-ND3', 'MT-CO1'],
    'Proliferating': ['MKI67', 'TOP2A', 'PCNA'],
    'Metabolic': ['PPARGC1A', 'CKMT2']
}

sc.pl.dotplot(
    adata_combined,
    marker_genes,
    groupby='leiden',
    save='_state_markers.png'
)

In [ ]:
cluster_annotations = {
    '2': 'Contractile_CM', '8': 'Contractile_CM', '10': 'Contractile_CM',
    '19': 'Proliferating_CM',
    '5': 'Mitochondrial_CM', '6': 'Mitochondrial_CM',
    '0': 'Immature_CM', '1': 'Immature_CM', '3': 'Immature_CM',
    '4': 'Immature_CM', '7': 'Immature_CM', '9': 'Immature_CM',
    '11': 'Immature_CM', '12': 'Immature_CM', '13': 'Immature_CM',
    '14': 'Immature_CM', '15': 'Immature_CM', '16': 'Immature_CM',
    '17': 'Immature_CM', '18': 'Immature_CM', '20': 'Immature_CM'
}

adata_combined.obs['cell_state'] = adata_combined.obs['leiden'].map(cluster_annotations)

sc.pl.umap(adata_combined,
    color=['cell_state', 'condition'],
    save='_annotated.png'
)

In [ ]:
# Quantify cell state proportions per condition
state_counts = adata_combined.obs.groupby(
    ['condition', 'cell_state']).size().unstack(fill_value=0)
state_props = state_counts.div(state_counts.sum(axis=1), axis=0) * 100
print(state_props.round(2))

In [ ]:
adata_combined.write('data/final_annotated.h5ad')
print("Final data saved!")